In [60]:
import pandas as pd
import os
from pathlib import Path
import csv

os.chdir('C:\\Users\\hamod4691\\Code\\arctic-benthic-biodiversity')

In [63]:
# Load the column data types specification
column_types_spec = pd.read_excel("column_data_types.xlsx")

# Load gbif_final.parquet
gbif_final_path = Path(r"C:\Users\hamod4691\Code\arctic-benthic-biodiversity\GBIF\gbif_final.parquet")
gbif_final = pd.read_parquet(gbif_final_path)

# Build a mapping of column -> target type from the specification
# Assuming column_types_spec has columns: column, dtype (or similar)
target_types = {}
for idx, row in column_types_spec.iterrows():
    col_name = row.get('column') or row.get('Column')
    col_type = row.get('final') or row.get('type') or row.get('data_type')
    if col_name and col_type:
        target_types[col_name] = str(col_type).strip().lower()

print(f"Loaded {len(target_types)} target types from column_data_types.xlsx")

# Define type conversion function
def cast_column(series, target_type):
    """Cast a series to the target type (using parquet-compatible types)"""
    target_type = str(target_type).strip().lower()
    
    # Normalize to parquet-compatible types
    if any(x in target_type for x in ['int32', 'int16', 'int8']):
        target_type = 'int64'
    elif target_type in ['int', 'integer', 'int64']:
        target_type = 'int64'
    elif any(x in target_type for x in ['float32', 'float16']):
        target_type = 'float64'
    elif target_type in ['float', 'float64', 'double']:
        target_type = 'float64'
    
    if target_type == 'string':
        return series.astype('string')
    elif target_type == 'datetime':
        return pd.to_datetime(series, errors='coerce')
    elif target_type == 'float64':
        return pd.to_numeric(series, errors='coerce').astype('float64')
    elif target_type == 'int64':
        return pd.to_numeric(series, errors='coerce').astype('Int64')  # Nullable Int64 handles NaNs
    elif target_type == 'boolean':
        return series.astype('boolean')
    elif target_type == 'category':
        return series.astype('category')
    else:
        return series

# Cast gbif_final columns to match target types
conversion_count = 0
for col in gbif_final.columns:
    if col in target_types:
        target = target_types[col]
        current_type = str(gbif_final[col].dtype).lower()
        
        if current_type != target:
            try:
                gbif_final[col] = cast_column(gbif_final[col], target)
                conversion_count += 1
                print(f"  {col}: {current_type} -> {target}")
            except Exception as e:
                print(f"  Warning: Could not convert {col} to {target}: {e}")

# Save the updated parquet (convert Int64 to int64 for parquet compatibility)
for col in gbif_final.columns:
    if gbif_final[col].dtype == 'Int64':
        gbif_final[col] = gbif_final[col].astype('int64', errors='ignore')
gbif_final.to_parquet(gbif_final_path, index=False)
print(f"\nConverted {conversion_count} columns in: {gbif_final_path}")

Loaded 62 target types from column_data_types.xlsx
  ID: float64 -> int32
  coordinateUncertaintyInMeters: float64 -> float32
  gebcoDepth: float64 -> float32
  startDayOfYear: float64 -> int32
  endDayOfYear: float64 -> int32
  year: float64 -> int32
  aphiaID: float64 -> int32
  areaSampled: float64 -> float32
  percentCoverage: float64 -> float32
  meshSize_mm: float64 -> float32
  sieveFraction_µm: float64 -> int32

Converted 11 columns in: C:\Users\hamod4691\Code\arctic-benthic-biodiversity\GBIF\gbif_final.parquet


In [64]:
gbif_final = pd.read_parquet("C:\\Users\\hamod4691\\Code\\arctic-benthic-biodiversity\\GBIF\\gbif_final.parquet")

# Reapply type conversions from column_data_types.xlsx (parquet doesn't preserve all pandas dtypes)
column_types_spec = pd.read_excel("column_data_types.xlsx")
target_types = {}
for idx, row in column_types_spec.iterrows():
    col_name = row.get('column') or row.get('Column')
    col_type = row.get('final') or row.get('type') or row.get('data_type')
    if col_name and col_type:
        target_types[col_name] = str(col_type).strip().lower()

def cast_column(series, target_type):
    """Cast a series to the target type (using parquet-compatible types)"""
    target_type = str(target_type).strip().lower()
    
    # Normalize to parquet-compatible types
    if any(x in target_type for x in ['int32', 'int16', 'int8']):
        target_type = 'int64'
    elif target_type in ['int', 'integer', 'int64']:
        target_type = 'int64'
    elif any(x in target_type for x in ['float32', 'float16']):
        target_type = 'float64'
    elif target_type in ['float', 'float64', 'double']:
        target_type = 'float64'
    
    if target_type == 'string':
        return series.astype('string')
    elif target_type == 'datetime':
        return pd.to_datetime(series, errors='coerce')
    elif target_type == 'float64':
        return pd.to_numeric(series, errors='coerce').astype('float64')
    elif target_type == 'int64':
        return pd.to_numeric(series, errors='coerce').astype('Int64')  # Nullable Int64 handles NaNs
    elif target_type == 'boolean':
        return series.astype('boolean')
    elif target_type == 'category':
        return series.astype('category')
    else:
        return series

for col in gbif_final.columns:
    if col in target_types:
        target = target_types[col]
        gbif_final[col] = cast_column(gbif_final[col], target)

# Column type overview
column_types = pd.DataFrame({
    "column": gbif_final.columns,
    "dtype": gbif_final.dtypes.astype(str).values,
    "non_null_count": gbif_final.notna().sum().values,
    "null_count": gbif_final.isna().sum().values,
    "non_null_pct": (gbif_final.notna().sum().values / len(gbif_final)) * 100,
})

column_types = column_types.sort_values(
    by=["non_null_count", "column"], ascending=[False, True]
 )

column_types.to_csv("gbif_final_column_types.csv", index=False)

column_types.head(30)

,column,dtype,non_null_count,null_count,non_null_pct
30,acceptedScientificName,string,2588475,0,100.000000
61,acceptedTaxonKey,Int64,2588475,0,100.000000
53,basisOfRecord,string,2588475,0,100.000000
38,class,string,2588475,0,100.000000
4,datasetKey,string,2588475,0,100.000000
7,decimalLatitudeStart,float64,2588475,0,100.000000
8,decimalLongitudeStart,float64,2588475,0,100.000000
19,eventID,string,2588475,0,100.000000
28,eventRemarks,string,2588475,0,100.000000
40,family,string,2588475,0,100.000000


In [65]:
# Diagnostic: Check actual types after reapplication
print("\n=== Checking converted columns ===")
converted_cols = ['ID', 'coordinateUncertaintyInMeters', 'gebcoDepth', 'startDayOfYear', 'endDayOfYear', 'year', 'aphiaID', 'areaSampled', 'percentCoverage', 'meshSize_mm', 'sieveFraction_µm']
for col in converted_cols:
    if col in gbif_final.columns:
        print(f"{col}: {gbif_final[col].dtype}")
    else:
        print(f"{col}: NOT FOUND")



=== Checking converted columns ===
ID: Int64
coordinateUncertaintyInMeters: float64
gebcoDepth: float64
startDayOfYear: Int64
endDayOfYear: Int64
year: Int64
aphiaID: Int64
areaSampled: float64
percentCoverage: float64
meshSize_mm: float64
sieveFraction_µm: Int64


In [ ]:
# Generate intelligent type suggestions for all columns
print("Generating type suggestions for all columns...\n")

type_suggestions = []

for col in gbif_final.columns:
    current_dtype = str(gbif_final[col].dtype)
    non_null_count = gbif_final[col].notna().sum()
    null_pct = (1 - non_null_count / len(gbif_final)) * 100
    
    # Infer suggested type based on content and name
    suggested_type = None
    
    # Check if it's numeric
    if current_dtype in ['int64', 'float64', 'Int64']:
        # Check column name for hints
        if any(x in col.lower() for x in ['id', 'key', 'count', 'year', 'day', 'month', 'aphia', 'gebco']):
            # Integer-like: ID, count, year, day, etc.
            if 'float' in current_dtype.lower() or '.' in str(gbif_final[col].dropna().iloc[0] if len(gbif_final[col].dropna()) > 0 else ''):
                suggested_type = 'float32'
            else:
                suggested_type = 'int32'
        elif any(x in col.lower() for x in ['depth', 'uncertainty', 'latitude', 'longitude', 'area', 'percentage', 'mesh', 'sieve', 'coordinate']):
            # Float-like: depth, coordinates, measurements
            suggested_type = 'float32'
        else:
            suggested_type = current_dtype.lower()
    
    # Check if it's datetime
    elif 'datetime' in current_dtype.lower():
        suggested_type = 'datetime'
    
    # Check if it's string/object
    elif current_dtype == 'object' or current_dtype == 'string':
        # Count unique values relative to total
        unique_ratio = gbif_final[col].nunique() / len(gbif_final)
        if unique_ratio < 0.05 and non_null_count > 100:
            suggested_type = 'category'
        else:
            suggested_type = 'string'
    
    # Check for boolean
    elif current_dtype == 'boolean':
        suggested_type = 'boolean'
    
    else:
        suggested_type = current_dtype
    
    type_suggestions.append({
        'column': col,
        'current': current_dtype,
        'final': suggested_type,
        'non_null_count': int(non_null_count),
        'null_pct': round(null_pct, 1)
    })

suggestions_df = pd.DataFrame(type_suggestions)

# Export to Excel
output_path = Path('column_data_types.xlsx')
with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
    suggestions_df.to_excel(writer, sheet_name='types', index=False)

print(f"Saved type suggestions to: {output_path}")
print(f"\nSample suggestions:")
print(suggestions_df[suggestions_df['final'] != suggestions_df['current']].head(20).to_string(index=False))


In [56]:
dataset_specs = {
    "obis_final": {
        "parquet": Path(r"C:\Users\hamod4691\Code\arctic-benthic-biodiversity\OBIS\obis_final.parquet"),
    },
    "gbif_final": {
        "parquet": Path(r"C:\Users\hamod4691\Code\arctic-benthic-biodiversity\GBIF\gbif_final.parquet"),
    },
    "ogsl_final": {
        "parquet": Path(r"C:\Users\hamod4691\Code\arctic-benthic-biodiversity\OGSL\ogsl_final.parquet"),
    },
    "cb_final": {
        "parquet": Path(r"C:\Users\hamod4691\Code\arctic-benthic-biodiversity\CB\cb_final.parquet"),
    },
}

def normalize_type(value):
    t = str(value).strip().lower()
    # integers (including pandas nullable Int64 etc.)
    if any(x in t for x in ["int", "integer", "smallint", "bigint"]) or any(x in t for x in ["int64", "int32", "int16"]):
        return "integer"
    # floats (including pandas nullable Float64)
    if any(x in t for x in ["float", "double", "decimal", "numeric", "real"]) or any(x in t for x in ["float64", "float32"]):
        return "float"
    # datetimes
    if any(x in t for x in ["datetime", "timestamp", "date", "time"]) or "datetime64" in t:
        return "datetime"
    # booleans
    if "bool" in t or t == "boolean":
        return "boolean"
    # categorical types
    if any(x in t for x in ["category", "categorical"]):
        return "category"
    # strings: treat object, str, pandas string, pyarrow string, etc. as the same
    if any(x in t for x in ["string", "object", "char", "text", "varchar", "str", "pyarrow.string", "string[python]", "string[pyarrow]"]):
        return "string"
    # binary/bytes
    if "bytes" in t or "binary" in t:
        return "binary"
    return t

# Load all parquets
parquets = {}
schema_rows = []
comparison_rows = []

for dataset, spec in dataset_specs.items():
    parquet_path = spec["parquet"]
    if not parquet_path.exists():
        schema_rows.append({"dataset": dataset, "status": "missing parquet"})
        continue
    
    df = pd.read_parquet(parquet_path)
    parquets[dataset] = df
    schema_rows.append({
        "dataset": dataset,
        "n_rows": len(df),
        "n_columns": len(df.columns),
        "status": "loaded"
    })

# Compare parquets to each other
datasets = list(parquets.keys())
all_columns = set()
for df in parquets.values():
    all_columns.update(df.columns)

for col in sorted(all_columns):
    for dataset1, dataset2 in [(datasets[i], datasets[j]) for i in range(len(datasets)) for j in range(i+1, len(datasets))]:
        df1 = parquets[dataset1]
        df2 = parquets[dataset2]
        
        in_df1 = col in df1.columns
        in_df2 = col in df2.columns
        
        if in_df1 and in_df2:
            type1 = normalize_type(str(df1[col].dtype))
            type2 = normalize_type(str(df2[col].dtype))
            if type1 != type2:
                comparison_rows.append({
                    "column": col,
                    "dataset1": dataset1,
                    "type1": str(df1[col].dtype),
                    "dataset2": dataset2,
                    "type2": str(df2[col].dtype),
                    "issue": "dtype mismatch"
                })
        elif in_df1 and not in_df2:
            comparison_rows.append({
                "column": col,
                "dataset1": dataset1,
                "dataset2": dataset2,
                "issue": f"present in {dataset1} only"
            })
        elif not in_df1 and in_df2:
            comparison_rows.append({
                "column": col,
                "dataset1": dataset1,
                "dataset2": dataset2,
                "issue": f"present in {dataset2} only"
            })

schema_report = pd.DataFrame(schema_rows)
comparison_report = pd.DataFrame(comparison_rows)

# Save to CSV
comparison_report.to_csv("parquet_comparison_report.csv", index=False)
schema_report.to_csv("parquet_schema_report.csv", index=False)

print(f"Saved comparison_report to: parquet_comparison_report.csv")
print(f"Saved schema_report to: parquet_schema_report.csv")

# Harmonise dtypes across dataframes and merge the four datasets
print("Harmonizing dtypes and merging datasets...")
# Determine target type for each column based on present types
target_types = {}
for col in sorted(all_columns):
    types = []
    for name, df in parquets.items():
        if col in df.columns:
            types.append(normalize_type(str(df[col].dtype)))
    if 'string' in types:
        target = 'string'
    elif 'datetime' in types:
        target = 'datetime'
    elif 'float' in types:
        target = 'float'
    elif 'integer' in types:
        target = 'integer'
    elif 'boolean' in types:
        target = 'boolean'
    elif 'category' in types:
        target = 'category'
    else:
        target = None
    target_types[col] = target

# Cast columns in each dataframe to the chosen target type (add missing columns)
for name, df in parquets.items():
    for col, target in target_types.items():
        if col not in df.columns:
            df[col] = pd.Series([pd.NA] * len(df), index=df.index)
        if target == 'string':
            df[col] = df[col].astype('string')
        elif target == 'datetime':
            df[col] = pd.to_datetime(df[col], errors='coerce')
        elif target == 'float':
            df[col] = pd.to_numeric(df[col], errors='coerce').astype('float64')
        elif target == 'integer':
            df[col] = pd.to_numeric(df[col], errors='coerce').astype('Int64')
        elif target == 'boolean':
            try:
                df[col] = df[col].astype('boolean')
            except Exception:
                df[col] = df[col].map({True: True, False: False}).astype('boolean')
        elif target == 'category':
            df[col] = df[col].astype('category')

# Concatenate in a consistent column order
col_order = sorted(all_columns)
merged = pd.concat([parquets[d] for d in datasets], ignore_index=True, sort=False)[col_order]

# Save merged dataset
merged_path = Path('All_data/merged_all.parquet')
merged.to_parquet(merged_path, index=False)
print(f"Saved merged to: {merged_path}")

schema_report, comparison_report, merged

Saved comparison_report to: parquet_comparison_report.csv
Saved schema_report to: parquet_schema_report.csv
Harmonizing dtypes and merging datasets...
Saved merged to: All_data\merged_all.parquet


(      dataset   n_rows  n_columns  status
 0  obis_final  1533558         62  loaded
 1  gbif_final  2588475         62  loaded
 2  ogsl_final     7257         62  loaded
 3    cb_final   243609         62  loaded,
               column    dataset1    type1    dataset2    type2           issue
 0                 ID  obis_final    Int32  gbif_final  float64  dtype mismatch
 1                 ID  gbif_final  float64  ogsl_final    Int32  dtype mismatch
 2                 ID  gbif_final  float64    cb_final    Int32  dtype mismatch
 3            aphiaID  obis_final    Int32  gbif_final  float64  dtype mismatch
 4            aphiaID  gbif_final  float64  ogsl_final    Int32  dtype mismatch
 5            aphiaID  gbif_final  float64    cb_final    Int32  dtype mismatch
 6       endDayOfYear  obis_final    Int32  gbif_final  float64  dtype mismatch
 7       endDayOfYear  gbif_final  float64  ogsl_final    Int32  dtype mismatch
 8       endDayOfYear  gbif_final  float64    cb_final    Int32 

In [3]:
# Calculate coverage (non-null percentage) for each column
coverage_data = []
for col in col_order:
    total_rows = len(merged)
    non_null_count = merged[col].notna().sum()
    coverage_pct = (non_null_count / total_rows) * 100
    coverage_data.append({
        "column": col,
        "non_null_count": non_null_count,
        "total_rows": total_rows,
        "coverage_percent": coverage_pct
    })

coverage_report = pd.DataFrame(coverage_data)
coverage_report.to_csv("column_coverage.csv", index=False)
print(f"Saved column coverage report to: column_coverage.csv")
print(coverage_report)

Saved column coverage report to: column_coverage.csv
                    column  non_null_count  total_rows  coverage_percent
0                       ID               0     4372899          0.000000
1   acceptedScientificName         4372899     4372899        100.000000
2         acceptedTaxonKey         2588475     4372899         59.193569
3                  aphiaID         4336707     4372899         99.172357
4              areaSampled          223537     4372899          5.111872
..                     ...             ...         ...               ...
57            verbatimDate         4249281     4372899         97.173088
58              verbatimID         4045497     4372899         92.512930
59  verbatimScientificName         4364974     4372899         99.818770
60               waterBody         3673932     4372899         84.015935
61                    year         4172079     4372899         95.407623

[62 rows x 4 columns]


# Deduplicate

Between-database deduplication

In [32]:
# Create a source priority mapping
source_priority = {
    'OGSL': 1,
    'CRITTERBASE': 2,
    'OBIS': 3,
    'GBIF': 4
}

# Deduplication key columns (we'll use rounded coords ≈50 m precision)
key_cols = ['year', 'startDayOfYear', 'decimalLongitudeStart_round', 'decimalLatitudeStart_round', 'aphiaID']

# --- Preprocess / normalize fields so "missing" is consistent ---
# Work on a copy and attach a stable row id for reliable keep/remove tracing
merged = merged.copy()
if '_row_id' not in merged.columns:
    merged['_row_id'] = merged.index.astype('int64')

# Normalize sourceDatabase (strip/case)
merged['sourceDatabase_norm'] = (
    merged['sourceDatabase'].astype('string')
          .str.strip()
          .str.upper()
)
merged['sourceDatabase_norm'] = merged['sourceDatabase_norm'].replace({'': pd.NA, 'NAN': pd.NA})

# Normalize numeric key columns (coerce invalid values to NaN)
for num_col in ['year', 'startDayOfYear', 'decimalLongitudeStart', 'decimalLatitudeStart', 'aphiaID']:
    if num_col in merged.columns:
        merged[num_col] = pd.to_numeric(merged[num_col], errors='coerce')

# Compute rounded coordinates at ~50 m precision
import numpy as np
# Approximate meters per degree latitude
_m_per_deg_lat = 111320.0
_lat_res_deg = 50.0 / _m_per_deg_lat  # ~0.000449 degrees (~50 m in latitude)

# For longitude, resolution depends on latitude (meters per degree longitude = 111320 * cos(lat))
# Avoid division by zero near poles by clipping cos(lat)
lat_rad = np.deg2rad(merged['decimalLatitudeStart'].astype('float').fillna(0.0))
_lon_res_deg = 50.0 / ( _m_per_deg_lat * np.clip(np.abs(np.cos(lat_rad)), 1e-6, 1.0) )

# Rounded latitude (same resolution for all rows)
merged['decimalLatitudeStart_round'] = (merged['decimalLatitudeStart'] / _lat_res_deg).round() * _lat_res_deg

# Rounded longitude (row-wise resolution)
merged['decimalLongitudeStart_round'] = (merged['decimalLongitudeStart'] / _lon_res_deg).round() * _lon_res_deg

# Identify rows with all key deduplication columns non-null (these are eligible for dedup)
required_numeric_cols = ['year', 'startDayOfYear', 'aphiaID']
required_text_cols = ['sourceDatabase_norm']
# plus the rounded coords must be present
numeric_complete = merged[required_numeric_cols].notna().all(axis=1)
coords_complete = merged[['decimalLongitudeStart_round', 'decimalLatitudeStart_round']].notna().all(axis=1)
text_complete = merged[required_text_cols].apply(
    lambda s: s.astype('string').str.strip().notna() & (s.astype('string').str.strip() != '')
).all(axis=1)
within_mask = numeric_complete & coords_complete & text_complete

# Split into rows to deduplicate and rows to keep as-is
to_dedup = merged[within_mask].copy()
to_keep = merged[~within_mask].copy()

# For each place+date combination, count how many distinct databases have it
place_date_db_count = to_dedup.groupby(['year', 'aphiaID', 'startDayOfYear', 'decimalLongitudeStart_round', 'decimalLatitudeStart_round'])['sourceDatabase_norm'].nunique().reset_index()
place_date_db_count.columns = ['year', 'aphiaID', 'startDayOfYear', 'decimalLongitudeStart_round', 'decimalLatitudeStart_round', 'db_count']

# Merge this count back into to_dedup
to_dedup = to_dedup.merge(place_date_db_count, on=['year', 'aphiaID', 'startDayOfYear', 'decimalLongitudeStart_round', 'decimalLatitudeStart_round'], how='left')

# Only rows where place+date appears in MULTIPLE databases are eligible for removal
cross_db_duplicates = to_dedup[to_dedup['db_count'] > 1].copy()
within_db_only = to_dedup[to_dedup['db_count'] == 1].copy()

if len(cross_db_duplicates) > 0:
    # For cross-database duplicates, keep all rows from the highest-priority database per place+date
    cross_db_duplicates['_priority'] = (
        cross_db_duplicates['sourceDatabase_norm'].map(source_priority).fillna(999)
    )
    # Keep sorted view for debugging readability
    cross_db_duplicates_sorted = cross_db_duplicates.sort_values(by=['year', 'aphiaID', 'startDayOfYear', 'decimalLongitudeStart_round', 'decimalLatitudeStart_round', '_priority'], na_position='last')

    # Compute best priority within each place+date group and keep all rows at that priority
    best_priority = cross_db_duplicates.groupby(['year', 'aphiaID', 'startDayOfYear', 'decimalLongitudeStart_round', 'decimalLatitudeStart_round'])['_priority'].transform('min')
    cross_db_duplicates_deduped = cross_db_duplicates[cross_db_duplicates['_priority'] == best_priority].copy()
else:
    cross_db_duplicates_deduped = cross_db_duplicates.copy()

# Within-database duplicates (same place/date in only one database) are kept as-is
within_db_only = within_db_only.copy()

# Compute which rows were removed during cross-database deduplication using stable row ids
if len(cross_db_duplicates) > 0:
    kept_row_ids = set(cross_db_duplicates_deduped['_row_id'].tolist())
    removed = cross_db_duplicates[~cross_db_duplicates['_row_id'].isin(kept_row_ids)].copy()
else:
    removed = cross_db_duplicates.copy()

# Drop helper columns before final concat/save
drop_cols = [c for c in ['_priority', 'db_count'] if c in cross_db_duplicates_deduped.columns]
cross_db_duplicates_deduped = cross_db_duplicates_deduped.drop(columns=drop_cols)
within_db_only = within_db_only.drop(columns=['db_count'] if 'db_count' in within_db_only.columns else [])

# Combine deduplicated cross-database records with within-database records
to_dedup_deduped = pd.concat([cross_db_duplicates_deduped, within_db_only], ignore_index=True)

# Concatenate back together
merged_deduped = pd.concat([to_dedup_deduped, to_keep], ignore_index=True)

# Do not persist helper row id
if '_row_id' in merged_deduped.columns:
    merged_deduped = merged_deduped.drop(columns=['_row_id'])

print(f"Original merged dataset: {len(merged)} rows")
print(f"After deduplication: {len(merged_deduped)} rows")
print(f"Rows deduplicated: {len(merged) - len(merged_deduped)}")
print(f"Cross-database duplicates found: {len(cross_db_duplicates)}")
print(f"Within-database records preserved: {len(within_db_only)}")

# Save the deduplicated dataset
merged_deduped.to_parquet('All_data/merged_all_deduplicated.parquet', index=False)
print(f"Saved deduplicated dataset to: All_data/merged_all_deduplicated.parquet")

if len(cross_db_duplicates) > 0:
    removed_counts = removed['sourceDatabase_norm'].value_counts(dropna=False).rename_axis('sourceDatabase').reset_index(name='removed_count')
    print('\nRemoved counts by sourceDatabase:')
    print(removed_counts)
    removed_counts.to_csv('dedup_removed_by_sourceDatabase.csv', index=False)
    print('Saved removed counts to: dedup_removed_by_sourceDatabase.csv')
    
    removed.head(50).to_parquet('All_data/dedup_removed_sample.parquet', index=False)
    print('Saved sample removed rows to: All_data/dedup_removed_sample.parquet')
else:
    print("\nNo cross-database duplicates found.")

Original merged dataset: 4372899 rows
After deduplication: 4040616 rows
Rows deduplicated: 332283
Cross-database duplicates found: 654282
Within-database records preserved: 2668791
Saved deduplicated dataset to: All_data/merged_all_deduplicated.parquet

Removed counts by sourceDatabase:
  sourceDatabase  removed_count
0           GBIF         309497
1           OBIS          22786
Saved removed counts to: dedup_removed_by_sourceDatabase.csv

Examples of removed rows (head):
     ID      acceptedScientificName  acceptedTaxonKey   aphiaID  areaSampled  \
119 NaN  Archypolycope semipunctata              <NA>  423158.0          NaN   
350 NaN                      Eualus              <NA>  106986.0          NaN   
439 NaN      Sclerochilus contortus              <NA>  127951.0          NaN   
441 NaN       Cytheropteron hamatum              <NA>  128017.0          NaN   
460 NaN    Microcythere medistriata              <NA>  463384.0          NaN   
690 NaN         Sclerocrangon ferox      

In [33]:
# Export between-source duplicates identified in the cross-database deduplication step
if 'cross_db_duplicates' not in globals() or 'cross_db_duplicates_deduped' not in globals():
    print("Run the between-database deduplication cell first.")
else:
    if 'key_cols' not in globals():
        key_cols = ['year', 'startDayOfYear', 'decimalLongitudeStart', 'decimalLatitudeStart']

    base = merged.copy() if 'merged' in globals() else pd.read_parquet('All_data/merged_all.parquet').copy()

    if '_row_id' not in base.columns:
        base['_row_id'] = base.index.astype('int64')

    if 'sourceDatabase_norm' not in base.columns:
        base['sourceDatabase_norm'] = (
            base['sourceDatabase']
            .astype('string')
            .str.strip()
            .str.upper()
            .replace({'': pd.NA, 'NAN': pd.NA})
        )

    kept_ids = set(cross_db_duplicates_deduped['_row_id']) if '_row_id' in cross_db_duplicates_deduped.columns else set()
    duplicate_ids = cross_db_duplicates['_row_id'] if '_row_id' in cross_db_duplicates.columns else pd.Index([])

    between_source_dedup = base[base['_row_id'].isin(duplicate_ids)].copy()
    between_source_dedup['between_source_dedup_status'] = (
        between_source_dedup['_row_id'].isin(kept_ids).map({True: 'Kept', False: 'Removed'})
    )

    sort_cols = key_cols + [c for c in ['between_source_dedup_status', 'sourceDatabase', 'sourceID', '_row_id'] if c in between_source_dedup.columns]
    between_source_dedup = between_source_dedup.sort_values(by=sort_cols, kind='mergesort', na_position='last')

    output_cols = [c for c in base.columns if c not in ['_row_id', 'sourceDatabase_norm']] + ['between_source_dedup_status']
    between_source_dedup = between_source_dedup[output_cols]

    output_path = Path('All_data/between_source_dedup.parquet')
    output_path.parent.mkdir(parents=True, exist_ok=True)
    between_source_dedup.to_parquet(output_path, index=False)

    print(f"Saved {len(between_source_dedup):,} rows to: {output_path}")
    print(between_source_dedup['between_source_dedup_status'].value_counts())

Saved 654,282 rows to: All_data\between_source_dedup.parquet
between_source_dedup_status
Removed    332283
Kept       321999
Name: count, dtype: int64


# Compare individualCount and gear presence between kept and removed records
kept_records = to_deduplicate_within_deduped
removed_records = removed_within

comparison_stats = []

for col in ['individualCount', 'gear']:
    kept_present = (
        kept_records[col]
        .astype('string')
        .str.strip()
        .replace({'': pd.NA})
        .notna()
        .sum()
    )
    kept_total = len(kept_records)
    kept_pct = (kept_present / kept_total) * 100
    
    removed_present = (
        removed_records[col]
        .astype('string')
        .str.strip()
        .replace({'': pd.NA})
        .notna()
        .sum()
    )
    removed_total = len(removed_records)
    removed_pct = (removed_present / removed_total) * 100
    
    comparison_stats.append({
        'column': col,
        'kept_present': kept_present,
        'kept_total': kept_total,
        'kept_percent': kept_pct,
        'removed_present': removed_present,
        'removed_total': removed_total,
        'removed_percent': removed_pct
    })

comparison_df = pd.DataFrame(comparison_stats)
print(comparison_df.to_string(index=False))
print("\nSummary:")
print(f"Kept records are more complete: {(comparison_df['kept_percent'] > comparison_df['removed_percent']).all()}")Within-database deduplication

In [27]:
# Remove within-database duplicates using aphiaID
# Prefer records that have values in individualCount, biomass, gear, and areaSampled
# when choosing the survivor for a duplicated within-database key.

# Load the deduplicated dataset
merged_deduped = pd.read_parquet('All_data/merged_all_dedup_between.parquet').copy()
if '_row_id' not in merged_deduped.columns:
    merged_deduped['_row_id'] = merged_deduped.index.astype('int64')

# Normalize sourceDatabase if not already done
if 'sourceDatabase_norm' not in merged_deduped.columns:
    merged_deduped['sourceDatabase_norm'] = (
        merged_deduped['sourceDatabase'].astype('string')
              .str.strip()
              .str.upper()
    )
    merged_deduped['sourceDatabase_norm'] = merged_deduped['sourceDatabase_norm'].replace({'': pd.NA, 'NAN': pd.NA})

# Define within-database deduplication columns
within_db_cols = ['sourceDatabase_norm', 'year', 'startDayOfYear', 'decimalLongitudeStart', 'decimalLatitudeStart', 'aphiaID']

# Only rows with all relevant columns populated are eligible for within-database dedup
required_numeric_cols = ['year', 'startDayOfYear', 'decimalLongitudeStart', 'decimalLatitudeStart']
required_text_cols = ['sourceDatabase_norm', 'aphiaID']

numeric_complete = merged_deduped[required_numeric_cols].notna().all(axis=1)
text_complete = merged_deduped[required_text_cols].apply(
    lambda s: s.astype('string').str.strip().notna() & (s.astype('string').str.strip() != '')
).all(axis=1)
within_db_mask = numeric_complete & text_complete

# Split into rows eligible for within-database dedup and rows to keep as-is
to_deduplicate_within = merged_deduped[within_db_mask].copy()
to_keep_within = merged_deduped[~within_db_mask].copy()

# Prefer records that have values in individualCount, biomass, gear, and areaSampled
# when choosing the survivor for a duplicated within-database key.
preference_cols = ['individualCount', 'biomass', 'gear', 'areaSampled']
preference_present_cols = [col for col in preference_cols if col in to_deduplicate_within.columns]
preference_scores = pd.Series(0, index=to_deduplicate_within.index, dtype='int64')
for col in preference_present_cols:
    present = (
        to_deduplicate_within[col]
        .astype('string')
        .str.strip()
        .replace({'': pd.NA})
        .notna()
        .astype('int64')
    )
    preference_scores = preference_scores.add(present, fill_value=0).astype('int64')

to_deduplicate_within['_preference_score'] = preference_scores

# Remove within-database duplicates (same sourceDatabase, place, date, and aphiaID)
# Keep the row with the most preferred observation fields; ties fall back to first occurrence after stable sorting
sort_cols = ['_preference_score', '_row_id']
sort_ascending = [True] * len(within_db_cols) + [False, True]
to_deduplicate_within = to_deduplicate_within.sort_values(
    by=within_db_cols + sort_cols,
    ascending=sort_ascending,
    kind='mergesort'
)
to_deduplicate_within_deduped = to_deduplicate_within.drop_duplicates(subset=within_db_cols, keep='first')

# Report what was removed via stable row id
kept_row_ids = set(to_deduplicate_within_deduped['_row_id'].tolist())
removed_within = to_deduplicate_within[~to_deduplicate_within['_row_id'].isin(kept_row_ids)].copy()

# Concatenate back together
to_deduplicate_within_deduped = to_deduplicate_within_deduped.drop(columns=['_preference_score'])
merged_within_deduped = pd.concat([to_deduplicate_within_deduped, to_keep_within], ignore_index=True)
if '_row_id' in merged_within_deduped.columns:
    merged_within_deduped = merged_within_deduped.drop(columns=['_row_id'])

print(f"Before within-database deduplication: {len(merged_deduped)} rows")
print(f"Eligible rows (all relevant columns present): {len(to_deduplicate_within)}")
print(f"After within-database deduplication: {len(merged_within_deduped)} rows")
print(f"Rows removed: {len(merged_deduped) - len(merged_within_deduped)}")

# Save the result
merged_within_deduped.to_parquet('All_data/merged_all_dedup_within.parquet', index=False)
print(f"Saved deduplicated dataset to: All_data/merged_all_dedup_within.parquet")

if len(removed_within) > 0:
    print(f"\nWithin-database duplicates removed by sourceDatabase:")
    print(removed_within['sourceDatabase_norm'].value_counts(dropna=False))
    print(f"\nExample removed rows:")
    print(removed_within.head(10)[['sourceDatabase_norm', 'year', 'startDayOfYear', 'decimalLongitudeStart', 'decimalLatitudeStart', 'aphiaID']])
else:
    print("\nNo within-database duplicates found.")

Before within-database deduplication: 2945953 rows
Eligible rows (all relevant columns present): 1978528
After within-database deduplication: 2945953 rows
Rows removed: 0
Saved deduplicated dataset to: All_data/merged_all_deduplicated.parquet

No within-database duplicates found.


In [ ]:
# Export OGSL and CRITTERBASE twins (removed + kept) to Excel with status labels
if 'removed_within' not in globals() or 'to_deduplicate_within_deduped' not in globals():
    print('Run the within-database deduplication cell first to create removed_within and to_deduplicate_within_deduped.')
else:
    key_cols_verify = ['sourceDatabase_norm', 'year', 'startDayOfYear', 'decimalLongitudeStart', 'decimalLatitudeStart', 'aphiaID']
    sources_to_export = ['OGSL', 'CRITTERBASE']

    all_twins = []

    for src in sources_to_export:
        rem_src = removed_within[removed_within['sourceDatabase_norm'] == src].copy()
        kept_src = to_deduplicate_within_deduped[to_deduplicate_within_deduped['sourceDatabase_norm'] == src].copy()

        if 'sourceID' not in rem_src.columns:
            rem_src['sourceID'] = pd.NA
        if 'sourceID' not in kept_src.columns:
            kept_src['sourceID'] = pd.NA

        export_cols = ['sourceID'] + key_cols_verify
        rem_src = rem_src[export_cols].copy()
        kept_src = kept_src[export_cols].copy()

        # Keep one representative kept row per key for matching
        kept_src_one = kept_src.drop_duplicates(subset=key_cols_verify, keep='first').copy()
        kept_src_one['status'] = 'Saved'

        rem_src['status'] = 'Removed'

        # Build pair IDs from exact key tuple so removed and kept twins are linkable
        rem_src['_pair_key'] = rem_src[key_cols_verify].astype('string').agg('||'.join, axis=1)
        kept_src_one['_pair_key'] = kept_src_one[key_cols_verify].astype('string').agg('||'.join, axis=1)

        matched_keys = set(rem_src['_pair_key']).intersection(set(kept_src_one['_pair_key']))

        rem_matched = rem_src[rem_src['_pair_key'].isin(matched_keys)].copy()
        kept_matched = kept_src_one[kept_src_one['_pair_key'].isin(matched_keys)].copy()

        rem_matched['source_group'] = src
        kept_matched['source_group'] = src

        rem_matched['pair_id'] = src + '_' + rem_matched['_pair_key'].astype('category').cat.codes.astype(str)
        key_to_pair = rem_matched.drop_duplicates('_pair_key').set_index('_pair_key')['pair_id'].to_dict()
        kept_matched['pair_id'] = kept_matched['_pair_key'].map(key_to_pair)

        out_cols = ['source_group', 'pair_id', 'status', 'sourceID'] + key_cols_verify
        all_twins.append(rem_matched[out_cols])
        all_twins.append(kept_matched[out_cols])

    twins_export = pd.concat(all_twins, ignore_index=True) if len(all_twins) > 0 else pd.DataFrame()
    twins_export = twins_export.sort_values(['source_group', 'pair_id', 'status']).reset_index(drop=True)

    output_path = Path('All_data/within_db_twins_ogsl_critterbase.xlsx')
    with pd.ExcelWriter(output_path) as writer:
        twins_export.to_excel(writer, sheet_name='twins', index=False)
        summary = twins_export.groupby(['source_group', 'status']).size().rename('rows').reset_index()
        summary.to_excel(writer, sheet_name='summary', index=False)

    print(f"Saved twin export to: {output_path}")
    print('Counts by source/status:')
    print(twins_export.groupby(['source_group', 'status']).size())

Saved twin export to: All_data\within_db_twins_ogsl_critterbase.xlsx
Counts by source/status:
source_group  status 
CRITTERBASE   Removed    56629
              Saved      30356
OGSL          Removed        3
              Saved          3
dtype: int64


In [30]:
# Diagnostic: Load saved removed rows and analyze dedup accuracy
# Check if removed rows are true duplicates or false positives

import pandas as pd
from pathlib import Path

# Load removed rows from cross-database dedup
try:
    removed_sample = pd.read_parquet('All_data/dedup_removed_sample.parquet')
    deduped = pd.read_parquet('All_data/merged_all_deduplicated.parquet')
    
    key_cols = ['year', 'startDayOfYear', 'decimalLongitudeStart', 'decimalLatitudeStart']
    obs_cols = ['sourceDatabase', 'aphiaID', 'sourceID', 'individualCount', 'biomass', 'gear', 'areaSampled', 'eventID', 'occurrenceID']
    obs_cols_present = [col for col in obs_cols if col in removed_sample.columns and col in deduped.columns]
    
    print(f"Analyzing {len(removed_sample)} removed rows (sample of 50)\n")
    
    # For each removed row, try to find its kept pair by matching key columns
    comparisons = []
    for idx, rem_row in removed_sample.iterrows():
        # Find rows in deduped that match this removed row's key
        key_match = deduped.copy()
        for col in key_cols:
            key_match = key_match[key_match[col] == rem_row[col]]
        
        if len(key_match) > 0:
            kept_row = key_match.iloc[0]
            
            # Count differing observation fields (excluding sourceDatabase since that's used for priority)
            diffs = []
            for col in obs_cols_present:
                if col == 'sourceDatabase':
                    continue
                rem_val = str(rem_row.get(col, pd.NA)).strip()
                kept_val = str(kept_row.get(col, pd.NA)).strip()
                rem_val_norm = 'NA' if rem_val.lower() in ['nan', 'na', 'none', ''] else rem_val
                kept_val_norm = 'NA' if kept_val.lower() in ['nan', 'na', 'none', ''] else kept_val
                
                if rem_val_norm != kept_val_norm and rem_val_norm != 'NA' and kept_val_norm != 'NA':
                    diffs.append(col)
            
            is_true_dup = len(diffs) == 0
            comparisons.append({
                'rem_db': rem_row.get('sourceDatabase', 'NA'),
                'kept_db': kept_row.get('sourceDatabase', 'NA'),
                'diffs': len(diffs),
                'fields': ', '.join(diffs) if diffs else 'none',
                'is_dup': is_true_dup,
                'rem_id': rem_row.get('sourceID', 'NA'),
                'kept_id': kept_row.get('sourceID', 'NA'),
            })
    
    comp_df = pd.DataFrame(comparisons)
    if len(comp_df) > 0:
        true_ct = comp_df['is_dup'].sum()
        false_ct = len(comp_df) - true_ct
        
        print(f"TRUE DUPLICATES: {true_ct}/{len(comp_df)} ({100*true_ct/len(comp_df):.0f}%)")
        print(f"FALSE POSITIVES: {false_ct}/{len(comp_df)} ({100*false_ct/len(comp_df):.0f}%)")
        print()
        
        # Load total counts
        removed_counts = pd.read_csv('dedup_removed_by_sourceDatabase.csv')
        total_removed = removed_counts['removed_count'].sum()
        false_rate = false_ct / len(comp_df)
        
        print(f"Extrapolated across all {total_removed} removed rows:")
        print(f"  ~{int(total_removed*(1-false_rate)):,} true duplicates ({100*(1-false_rate):.0f}%)")
        print(f"  ~{int(total_removed*false_rate):,} false positives ({100*false_rate:.0f}%)")
        print()
        
        print("Sample rows (10 examples):")
        print(comp_df[['rem_db', 'kept_db', 'diffs', 'fields', 'rem_id', 'kept_id']].head(10).to_string(index=False))
        
except FileNotFoundError as e:
    print(f"Run the cross-database dedup cell first: {e}")


Analyzing 50 removed rows (sample of 50)

TRUE DUPLICATES: 0/50 (0%)
FALSE POSITIVES: 50/50 (100%)

Extrapolated across all 302248 removed rows:
  ~0 true duplicates (0%)
  ~302,248 false positives (100%)

Sample rows (10 examples):
rem_db     kept_db  diffs                                            fields                               rem_id kept_id
  OBIS CRITTERBASE      5 aphiaID, sourceID, individualCount, gear, eventID 001c218b-0362-404e-9d4f-5128ab31ffdd  623188
  OBIS CRITTERBASE      4                  aphiaID, sourceID, gear, eventID 004d5197-5ba7-47ba-beb3-72a09ecfc94a  612777
  OBIS CRITTERBASE      5 aphiaID, sourceID, individualCount, gear, eventID 005e6a00-a6c2-4993-bd41-83ad0158281e  623188
  OBIS CRITTERBASE      5 aphiaID, sourceID, individualCount, gear, eventID 005eb50a-0f0a-43a8-b933-39f973959ac3  623188
  OBIS CRITTERBASE      5 aphiaID, sourceID, individualCount, gear, eventID 0063680b-a5ed-4948-bc89-658d68cf18a5  623188
  OBIS CRITTERBASE      4                

# Filter out irrelevant datasets

Remove datasets whose records were removed in the deduplication process

In [19]:
gbif_obis_path = Path("GBIF/datasets_GBIF_OBIS.parquet")
merged_path = Path("All_data/merged_all_deduplicated.parquet")
output_path = Path("GBIF/datasets_GBIF_OBIS_filtered.parquet")

gbif_obis = pd.read_parquet(gbif_obis_path)

if "merged_deduped" in globals():
    merged_source = merged_deduped
else:
    merged_source = pd.read_parquet(merged_path)

def _norm_series(s):
    return (
        s.astype("string")
         .str.strip()
         .replace({"": pd.NA, "nan": pd.NA, "None": pd.NA, "NA": pd.NA, "N/A": pd.NA})
    )

valid_dataset_keys = set(_norm_series(merged_source["datasetKey"]).dropna().unique())
alt_ids = _norm_series(gbif_obis["alternateIdentifier"])

keep_mask = alt_ids.isin(valid_dataset_keys)
filtered = gbif_obis.loc[keep_mask].copy()

removed_count = len(gbif_obis) - len(filtered)
print(f"Original rows: {len(gbif_obis)}")
print(f"Removed rows: {removed_count}")
print(f"Remaining rows: {len(filtered)}")

filtered.to_parquet(output_path, index=False)
print(f"Saved filtered file to: {output_path}")

Original rows: 617
Removed rows: 16
Remaining rows: 601
Saved filtered file to: GBIF\datasets_GBIF_OBIS_filtered.parquet


In [20]:
manual_citations_path = Path("OBIS/unique_bibliographic_citations_manual.csv")
filtered_path = Path("GBIF/datasets_GBIF_OBIS_filtered.parquet")

# Load the manual citations file
manual_citations = pd.read_csv(manual_citations_path)

# Load the filtered datasets
filtered = pd.read_parquet(filtered_path)

# Normalize citation columns for matching
manual_citations['bibliographicCitation_norm'] = _norm_series(manual_citations['bibliographicCitation'])
filtered['citation_norm'] = _norm_series(filtered['citation'])

# Merge the Keep column from manual_citations
filtered = filtered.merge(
    manual_citations[['bibliographicCitation_norm', 'keep']],
    left_on='citation_norm',
    right_on='bibliographicCitation_norm',
    how='left'
)

# Merge any keep-like columns into one canonical Keep column
keep_like_cols = [col for col in ['Keep', 'Keep_x', 'Keep_y', 'keep'] if col in filtered.columns]
if keep_like_cols:
    canonical_keep = filtered[keep_like_cols[0]].copy()
    for col in keep_like_cols[1:]:
        canonical_keep = canonical_keep.combine_first(filtered[col])
    filtered['Keep'] = canonical_keep
    filtered = filtered.drop(columns=[col for col in keep_like_cols if col != 'Keep'])

# Drop the temporary normalized columns
filtered = filtered.drop(columns=['citation_norm', 'bibliographicCitation_norm'])

# Count missing Keep entries
missing_keep = filtered['Keep'].isna().sum()
print(f"Records lacking a Keep entry: {missing_keep}")

# Save back to the same location

filtered.to_parquet(filtered_path, index=False)
print(f"Saved updated file to: {filtered_path}")

Records lacking a Keep entry: 599
Saved updated file to: GBIF\datasets_GBIF_OBIS_filtered.parquet


Add Keep column to records file

In [22]:
keep_manual_path = Path("GBIF/merged_dataset_with_keep_manual.csv")
filtered_path = Path("GBIF/datasets_GBIF_OBIS_filtered.parquet")
output_path = Path("GBIF/datasets_GBIF_OBIS_filtered_with_keep.xlsx")

# Load the keep manual file
keep_manual = pd.read_csv(keep_manual_path)

# Normalize datasetKey in keep_manual for matching
keep_manual['datasetKey_norm'] = _norm_series(keep_manual['datasetKey'])

# Normalize datasetKey in filtered for matching
filtered['datasetKey_norm'] = _norm_series(filtered['alternateIdentifier'])

# Merge the Keep column
filtered = filtered.merge(
    keep_manual[['datasetKey_norm', 'Keep']],
    on='datasetKey_norm',
    how='left'
)

# Merge any keep-like columns into one canonical Keep column
keep_like_cols = [col for col in ['Keep', 'Keep_x', 'Keep_y', 'keep'] if col in filtered.columns]
if keep_like_cols:
    canonical_keep = filtered[keep_like_cols[0]].copy()
    for col in keep_like_cols[1:]:
        canonical_keep = canonical_keep.combine_first(filtered[col])
    filtered['Keep'] = canonical_keep
    filtered = filtered.drop(columns=[col for col in keep_like_cols if col != 'Keep'])

# Count missing Keep entries
missing_keep = filtered['Keep'].isna().sum()
print(f"Datasets lacking a Keep entry: {missing_keep}")

# Drop the temporary normalized column
filtered = filtered.drop(columns=['datasetKey_norm'])

# Save back to the same location
filtered.to_parquet(filtered_path, index=False)
print(f"Saved updated file to: {filtered_path}")

# Save to new location
filtered.to_excel(output_path, index=False)
print(f"Saved updated file to: {output_path}")

Datasets lacking a Keep entry: 355
Saved updated file to: GBIF\datasets_GBIF_OBIS_filtered.parquet
Saved updated file to: GBIF\datasets_GBIF_OBIS_filtered_with_keep.xlsx


Go through datasets with missing classification manually, save as datasets_GBIF_OBIS_manual.parquet

In [24]:
# Load the datasets
manual_path = Path("GBIF/datasets_GBIF_OBIS_manual.xlsx")
merged_path = Path("All_data/merged_all_deduplicated.parquet")

manual = pd.read_excel(manual_path)
merged_deduped = pd.read_parquet(merged_path)

# Normalize datasetKey in both for matching
manual['datasetKey_norm'] = _norm_series(manual['alternateIdentifier'])
merged_deduped['datasetKey_norm'] = _norm_series(merged_deduped['datasetKey'])

# Merge the Keep column from manual
merged_deduped = merged_deduped.merge(
    manual[['datasetKey_norm', 'Keep']],
    on='datasetKey_norm',
    how='left'
)

# Drop the temporary normalized column
merged_deduped = merged_deduped.drop(columns=['datasetKey_norm'])

# Save back to the same location
merged_deduped.to_parquet(merged_path, index=False)
print(f"Saved updated file to: {merged_path}")

Saved updated file to: All_data\merged_all_deduplicated.parquet


Remove records with R or U

In [25]:
# Load the deduplicated dataset
merged_deduped = pd.read_parquet(merged_path)

# Filter out rows where Keep column contains 'R' or 'U'
merged_filtered = merged_deduped[~merged_deduped['Keep'].isin(['R', 'U'])].copy()

print(f"Original rows: {len(merged_deduped)}")
print(f"Removed rows (Keep='R' or 'U'): {len(merged_deduped) - len(merged_filtered)}")
print(f"Remaining rows: {len(merged_filtered)}")

# Save back to the same location
merged_filtered.to_parquet(merged_path, index=False)
print(f"Saved filtered file to: {merged_path}")

Original rows: 3190400
Removed rows (Keep='R' or 'U'): 244447
Remaining rows: 2945953
Saved filtered file to: All_data\merged_all_deduplicated.parquet
